# L05 -- Multilayer Perceptron

Companion notebook for L05. Reproduces the 3-step PyTorch training pipeline from the slides, with a class-imbalance experiment at the end.

**Prerequisite**: micromamba `pdpower` env. Slides: `L05_mlp.pdf`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, confusion_matrix, auc
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
print(f"torch {torch.__version__}")

## 1. Data and model

`make_moons` is the ML 101 nonlinear-classification dataset. We split 75/25 train/val.

In [ ]:
X, y = make_moons(n_samples=400, noise=0.20, random_state=0)
X_tr_np, X_va_np, y_tr_np, y_va_np = train_test_split(X, y, test_size=0.25, random_state=0)
X_tr = torch.tensor(X_tr_np, dtype=torch.float32)
y_tr = torch.tensor(y_tr_np, dtype=torch.float32).unsqueeze(1)
X_va = torch.tensor(X_va_np, dtype=torch.float32)
y_va = torch.tensor(y_va_np, dtype=torch.float32).unsqueeze(1)

model = nn.Sequential(
    nn.Linear(2, 32), nn.ReLU(),
    nn.Linear(32, 32), nn.ReLU(),
    nn.Linear(32, 1),  nn.Sigmoid(),
)
print(model)

## 2. Training loop with early stopping

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
bce = nn.BCELoss()
best_val, best_state = float("inf"), None
tr_losses, va_losses = [], []
for epoch in range(200):
    model.train(); opt.zero_grad()
    loss = bce(model(X_tr), y_tr); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        v = bce(model(X_va), y_va).item()
    tr_losses.append(loss.item()); va_losses.append(v)
    if v < best_val:
        best_val = v
        best_state = {k: t.clone() for k, t in model.state_dict().items()}

model.load_state_dict(best_state)
print(f"Best validation BCE: {best_val:.4f} at epoch {int(np.argmin(va_losses))}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(tr_losses, label="train"); ax.plot(va_losses, label="val")
ax.axvline(np.argmin(va_losses), color="gray", linestyle="--", alpha=0.5, label="early-stop")
ax.set_xlabel("epoch"); ax.set_ylabel("BCE"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Decision threshold and confusion matrix

We pick the threshold maximizing recall subject to precision $\geq 0.85$ -- the L05 fix discussed in the lecture.

In [ ]:
with torch.no_grad():
    p_va = model(X_va).squeeze().numpy()

prec, rec, thr = precision_recall_curve(y_va.squeeze().numpy(), p_va)
valid = prec[:-1] >= 0.85                                # aligned with thr
if valid.any():
    threshold = thr[valid][np.argmax(rec[:-1][valid])]
else:
    threshold = 0.5

y_pred = (p_va >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_va.squeeze().numpy(), y_pred).ravel()
precision = tp / (tp + fp + 1e-9)
recall    = tp / (tp + fn + 1e-9)
auprc     = auc(rec, prec)
print(f"threshold={threshold:.3f}  recall={recall:.3f}  precision={precision:.3f}  AUPRC={auprc:.3f}")
print(f"Confusion matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")

## 4. Decision boundary

In [ ]:
xx, yy = np.meshgrid(np.linspace(-2.0, 3.0, 200), np.linspace(-1.0, 2.0, 200))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad():
    zz = model(grid).numpy().reshape(xx.shape)

fig, ax = plt.subplots(figsize=(6, 4))
ax.contourf(xx, yy, zz, levels=20, cmap="RdBu_r", alpha=0.6)
ax.contour(xx, yy, zz, levels=[0.5], colors="black", linewidths=2)
ax.scatter(X_tr_np[:,0], X_tr_np[:,1], c=y_tr_np, cmap="bwr", edgecolor="k", s=20)
ax.set_title("MLP decision boundary"); plt.tight_layout(); plt.show()

## 5. Class imbalance experiment

Keep all positives, drop 80% of negatives. Train plain BCE vs class-weighted BCE; compare recall.

In [ ]:
mask = (y_tr_np == 0) | (np.random.rand(len(y_tr_np)) > 0.8)
X_tr_imb = torch.tensor(X_tr_np[mask], dtype=torch.float32)
y_tr_imb = torch.tensor(y_tr_np[mask], dtype=torch.float32).unsqueeze(1)
print(f"Imbalanced training set: {len(y_tr_imb)} total, {int(y_tr_imb.sum())} positive")

def train_mlp(X, y, loss_fn, epochs=200):
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(2,32), nn.ReLU(),
                      nn.Linear(32,32), nn.ReLU(),
                      nn.Linear(32,1),  nn.Sigmoid())
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)
    for _ in range(epochs):
        opt.zero_grad(); loss_fn(m(X), y).backward(); opt.step()
    return m

def recall_at_05(m, X, y):
    with torch.no_grad(): p = m(X).squeeze().numpy()
    pred = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y.squeeze().numpy(), pred).ravel()
    return tp / (tp + fn + 1e-9)

plain = train_mlp(X_tr_imb, y_tr_imb, nn.BCELoss())
weighted_bce = nn.BCELoss(weight=torch.where(y_tr_imb == 1, 5.0, 1.0))
weighted = train_mlp(X_tr_imb, y_tr_imb, weighted_bce)

print(f"Plain BCE        recall on val: {recall_at_05(plain,    X_va, y_va):.3f}")
print(f"Class-weighted   recall on val: {recall_at_05(weighted, X_va, y_va):.3f}")

## Homework

### Required
1. Repeat the training with seeds `{0, 1, 2, 3, 4}`. Report mean and stdev of validation recall.
2. Add a third hidden layer. Re-train. Does anything change?
3. Plot the decision boundary for the class-imbalanced model. How does it differ from the balanced-data boundary?

### Optional
- Replace ReLU with `nn.Tanh()`. Compare convergence speed and final accuracy.
- Implement focal loss and compare to class-weighted BCE.

In [ ]:
# TODO Q1: seed sweep
# ...

# TODO Q2: third hidden layer
# ...

# TODO Q3: imbalanced boundary plot
# ...